# Data Cleaning & Exploration

## Setup Notebook

### Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Import Data

In [ ]:
df = pd.read_csv('datasets/cfb_box-scores_2002-2025.csv', parse_dates=['date'])

## Clean The Data

### Set Flags for Team Ranking and Home Team Win

In [ ]:
# 1 if the team was ranked going into the game, 0 if not (blank rank = unranked)
df['home_is_ranked'] = df['rank_home'].notna().astype(int)
df['away_is_ranked'] = df['rank_away'].notna().astype(int)

# Keep the rank as a number, unranked teams placed below #25
df['rank_home_filled'] = df['rank_home'].fillna(30)
df['rank_away_filled'] = df['rank_away'].fillna(30)

# Target: 1 if the home team won, 0 if the away team won
df['home_win'] = (df['score_home'] > df['score_away']).astype(int)

### Make a DF with only FBS Teams (we are only wanting to predict FBS matchups)

In [ ]:
# A blank conference means that team wasn't FBS that season
df_fbs = df.dropna(subset=['conf_home', 'conf_away']).copy()

print(len(df), '->', len(df_fbs))

### Make `neutral` '0/1' instead of 'True/False'

In [ ]:
df_fbs['neutral'] = df_fbs['neutral'].astype(int)

### Drop Unnecessary Columns and By Quarter In-Game Stats Columns

In [ ]:
quarter_cols = [
	f'{q}_{side}'
	for q in ['q1', 'q2', 'q3', 'q4', 'ot']
	for side in ['home', 'away']
]

df_fbs = df_fbs.drop(columns=['possession_home', 'possession_away', 'tv', 'attendance', 'rank_away', 'rank_home'] + quarter_cols)

### Get In-Conference Games

In [ ]:
df_fbs['conf_game'] = ((df_fbs['conf_home'] == df_fbs['conf_away']) & (df_fbs['conf_home'] != 'ind')).astype(int)

### One Hot Encode Conference

In [ ]:
df_fbs = pd.get_dummies(df_fbs, columns=['conf_home', 'conf_away'], dtype=int)

### Export Cleaned Data

In [ ]:
df_fbs.to_csv('datasets/cfb_fbs_clean.csv', index=False)

## Explore the Cleaned Data

In [ ]:
df_fbs.describe().T                                           # ranges and averages for every number column
df_fbs['home_win'].mean()                                     # overall home win rate
df_fbs.groupby('season')['home_win'].agg(home_win_pct='mean', home_wins='count').T # win rate and game count by season

### Home win rate by season

In [ ]:
win_by_season = df_fbs.groupby(['season', 'neutral'])['home_win'].mean().unstack()
win_by_season.columns = ['Home Field', 'Neutral Site']
win_by_season.columns.name = 'Venue'
win_by_season.plot(marker='o', figsize=(10, 4))
plt.axhline(0.5, color='gray', linestyle='--')
plt.ylabel('Home team win rate')
plt.title('Home win rate by season')
plt.show()

### How Close Are Games?

In [ ]:
margin = df_fbs['score_home'] - df_fbs['score_away']
p = sns.histplot(margin, bins=50)
plt.axvline(0, color='red')
plt.xlabel('Home margin (home score - away score)')
plt.title('Final score margins')
plt.show()

### Which game stats go with winning?

In [ ]:
stats = ['total_yards', 'rush_yards', 'pass_yards', 'first_downs', 'int', 'fum', 'pen_yards']
diffs = pd.DataFrame({s: df_fbs[f'{s}_home'] - df_fbs[f'{s}_away'] for s in stats})
diffs.corrwith(df_fbs['home_win']).sort_values().plot.barh()
plt.title('Correlation of home-minus-away stats with home win')
plt.show()

### How do 4th downs and 3rd downs affect home win rate?

In [ ]:
cols = ['fourth_down_att_home', 'fourth_down_comp_home',
        'third_down_att_home', 'third_down_comp_home', 'home_win']
downs = df_fbs[cols].dropna().copy()

# Group the rare values at each end so every bar has enough games behind it
downs['4th_att'] = downs['fourth_down_att_home'].clip(upper=6).astype(int)     # 6 = "6 or more"
downs['4th_conv'] = downs['fourth_down_comp_home'].clip(upper=4).astype(int)   # 4 = "4 or more"
downs['3rd_att'] = downs['third_down_att_home'].clip(8, 20).astype(int)        # "8 or fewer" to "20 or more"
downs['3rd_conv'] = downs['third_down_comp_home'].clip(1, 11).astype(int)      # "1 or fewer" to "11 or more"

panels = [
    ('4th_att', 'Home win rate by 4th down attempts'),
    ('4th_conv', 'Home win rate by successful 4th downs'),
    ('3rd_att', 'Home win rate by 3rd down attempts'),
    ('3rd_conv', 'Home win rate by successful 3rd downs'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharey=True)
for ax, (col, title) in zip(axes.flat, panels):
    sns.barplot(data=downs, x=col, y='home_win', ax=ax)
    ax.axhline(downs['home_win'].mean(), color='gray', linestyle='--')  # overall average
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Home win rate')
plt.tight_layout()
plt.show()

print(downs[cols].corr()['home_win'])